### RAG_Pipelines - Data Ingestion to Vector DB

In [6]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

### Read all txt files inside the directory

def process_all_txts(txt_directory):
    """Process all TXT files in a directory"""

    all_documents = []

    txt_dir = Path(txt_directory)

    # Find all TXT files recursively
    txt_files = list(txt_dir.glob("**/*.txt"))

    print(f"Found {len(txt_files)} TXT files to process")

    for txt_file in txt_files:

        print(f"\nProcessing: {txt_file.name}")

        try:
            loader = TextLoader(str(txt_file), encoding="utf-8")

            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = txt_file.name
                doc.metadata["file_type"] = "txt"

            all_documents.extend(documents)

        except Exception as e:
            print(f"Error processing {txt_file.name}: {e}")

    return all_documents


documents = process_all_txts("../data/policy_files")

print(f"\nTotal documents loaded: {len(documents)}")

Found 5 TXT files to process

Processing: customer_communication_policy.txt

Processing: data_privacy_policy.txt

Processing: hr_ethics_policy.txt

Processing: information_security_policy.txt

Processing: legal_marketing_policy.txt

Total documents loaded: 5


In [7]:
!pip install langchain-text-splitters


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Text splitting into chunks

def split_documents(documents, chunk_size=300, chunk_overlap=50):

    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example chunk
    if split_docs:
        print("\nExample chunk:\n")
        print(split_docs[0].page_content[:300])
        print("\nMetadata:")
        print(split_docs[0].metadata)

    return split_docs




In [9]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

dir_loader = DirectoryLoader(
    "../data/policy_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

documents

100%|██████████| 5/5 [00:00<00:00, 2751.81it/s]


[Document(metadata={'source': '..\\data\\policy_files\\customer_communication_policy.txt'}, page_content='\nPOLICY CATEGORY 5 — CUSTOMER COMMUNICATION & OPERATIONS POLICY\n\nPolicy 5.1 — Professional Customer Communication\n\nEmployees must maintain respectful, professional, and accurate communication with customers at all times.\n\nPolicy 5.2 — Unauthorized Commitments\n\nEmployees must not provide delivery guarantees, pricing commitments, refunds, or operational promises without authorized approval.\n\nPolicy 5.3 — Confidential Project Discussions\n\nInternal project details, unreleased features, roadmap discussions, and operational strategies must not be disclosed externally.\n\nPolicy 5.4 — Escalation Requirements\n\nCustomer complaints involving legal, financial, or security concerns must be escalated to the designated compliance team.\n\nPolicy 5.5 — Communication Record Retention\n\nBusiness-critical communication with customers and stakeholders must be retained according to org

In [10]:
chunks = split_documents(documents)
chunks

Split 5 documents into 34 chunks

Example chunk:

POLICY CATEGORY 5 — CUSTOMER COMMUNICATION & OPERATIONS POLICY

Policy 5.1 — Professional Customer Communication

Employees must maintain respectful, professional, and accurate communication with customers at all times.

Policy 5.2 — Unauthorized Commitments

Metadata:
{'source': '..\\data\\policy_files\\customer_communication_policy.txt'}


[Document(metadata={'source': '..\\data\\policy_files\\customer_communication_policy.txt'}, page_content='POLICY CATEGORY 5 — CUSTOMER COMMUNICATION & OPERATIONS POLICY\n\nPolicy 5.1 — Professional Customer Communication\n\nEmployees must maintain respectful, professional, and accurate communication with customers at all times.\n\nPolicy 5.2 — Unauthorized Commitments'),
 Document(metadata={'source': '..\\data\\policy_files\\customer_communication_policy.txt'}, page_content='Policy 5.2 — Unauthorized Commitments\n\nEmployees must not provide delivery guarantees, pricing commitments, refunds, or operational promises without authorized approval.\n\nPolicy 5.3 — Confidential Project Discussions'),
 Document(metadata={'source': '..\\data\\policy_files\\customer_communication_policy.txt'}, page_content='Policy 5.3 — Confidential Project Discussions\n\nInternal project details, unreleased features, roadmap discussions, and operational strategies must not be disclosed externally.\n\nPolicy 5.

### Embedding and VectorDB

In [11]:
# CELL 1 — IMPORTS

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid

from typing import List
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# EMBEDDING MANAGER + GENERATE EMBEDDINGS + INITIALIZATION

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None

        self._load_model()

    def _load_model(self):

        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:

            print(f"Error loading model {self.model_name}: {e}")

            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """Generate embeddings for a list of texts"""

        if not self.model:

            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize embedding manager
embedding_manager = EmbeddingManager()

# Create text list from chunks
chunk_texts = [doc.page_content for doc in chunks]

print(f"\nTotal chunk texts: {len(chunk_texts)}")

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(chunk_texts)

print("\nEmbedding Shape:", embeddings.shape)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2828.74it/s]
C:\Users\HP\AppData\Local\Temp\ipykernel_10876\3367072354.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


Model loaded successfully. Embedding dimension: 384

Total chunk texts: 34
Generating embeddings for 34 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.03it/s]

Generated embeddings with shape: (34, 384)

Embedding Shape: (34, 384)


In [13]:
# VECTOR STORE USING CHROMADB

import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "policy_documents",
        persist_directory: str = "../data/vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):

        """Initialize ChromaDB client and collection"""

        try:

            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Policy document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. Collection: {self.collection.name}"
            )

            print(
                f"Existing documents in collection: {self.collection.count()}"
            )

        except Exception as e:

            print(f"Error initializing vector store: {e}")

            raise

    def add_documents(self, documents: List, embeddings: np.ndarray):

        """
        Add documents and embeddings to vector store
        """

        if len(documents) != len(embeddings):

            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"

            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB
        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents to vector store"
            )

            print(
                f"Total documents in collection: {self.collection.count()}"
            )

        except Exception as e:

            print(f"Error adding documents to vector store: {e}")

            raise


# Initialize vector store
vectorstore = VectorStore()

# Add chunks + embeddings to vector DB
vectorstore.add_documents(chunks, embeddings)

vectorstore

Vector store initialized. Collection: policy_documents
Existing documents in collection: 34
Adding 34 documents to vector store...
Successfully added 34 documents to vector store
Total documents in collection: 68


## Retrival Pipeline

In [14]:
# RETRIEVER PIPELINE

from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from vector store"""

    def __init__(self, vector_store, embedding_manager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5
    ):

        print(f"\nRetrieving documents for query:\n{query}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector DB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results.get("distances", [[0]*len(documents)])[0]
            ids = results["ids"][0]

            for i, (
                doc_id,
                document,
                metadata,
                distance
            ) in enumerate(
                zip(ids, documents, metadatas, distances)
            ):

                retrieved_docs.append(
                    {
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": distance,
                        "rank": i + 1
                    }
                )

        print(f"\nRetrieved {len(retrieved_docs)} documents")

        # DISPLAY RESULTS AUTOMATICALLY
        for result in retrieved_docs:

            print("\n" + "="*80)

            print(f"Rank: {result['rank']}")

            print(f"Similarity Score: {result['similarity_score']:.4f}")

            print("\nRetrieved Policy:\n")

            print(result["content"])

            print("\nMetadata:")

            print(result["metadata"])

        return retrieved_docs

In [15]:
# Initialize Retriever

rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

rag_retriever

In [16]:
query = """
I will send customer Aadhaar and banking information
through personal email for quick approval.
"""

results = rag_retriever.retrieve(query)


Retrieving documents for query:

I will send customer Aadhaar and banking information
through personal email for quick approval.

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.74it/s]

Generated embeddings with shape: (1, 384)

Retrieved 5 documents

Rank: 1
Similarity Score: 1.1524

Retrieved Policy:

Employees must not share customer PAN, Aadhaar, passport numbers, banking details, or other personally identifiable information (PII) through unsecured communication channels such as WhatsApp, personal email, SMS, or public chat platforms.

Policy 1.2 — Authorized Data Sharing

Metadata:
{'source': '..\\data\\policy_files\\data_privacy_policy.txt', 'content_length': 277, 'doc_index': 8}

Rank: 2
Similarity Score: 1.1524

Retrieved Policy:

Employees must not share customer PAN, Aadhaar, passport numbers, banking details, or other personally identifiable information (PII) through unsecured communication channels such as WhatsApp, personal email, SMS, or public chat platforms.

Policy 1.2 — Authorized Data Sharing

Metadata:
{'source': '..\\data\\policy_files\\data_privacy_policy.txt', 'content_length': 277, 'doc_index': 8}

Rank: 3
Similarity Score: 1.4235

Retrieved Po

### Integration Context Vectordb pipeline with LLM Output

In [31]:
# =========================
# LLM + RAG PIPELINE
# =========================

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get API Key from .env
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

# =========================
# SIMPLE RAG FUNCTION
# =========================

def rag_simple(
    query,
    retriever,
    llm,
    top_k=3
):

    # Retrieve relevant policies
    results = retriever.retrieve(
        query=query,
        top_k=top_k
    )

    # Create context from retrieved docs
    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    # If no context found
    if not context:
        return "No relevant policy found."

    # Prompt
    prompt = f"""
You are an AI Compliance Assistant.

Analyze the employee statement using the company policies.

Retrieved Policies:
{context}

Employee Statement:
{query}

Tasks:
1. Detect whether policy violation exists
2. Explain the violation clearly
3. Mention which policy is violated
4. Suggest safer compliant behavior

Return output in this format:

Violation Detected:
Violated Policy:
Explanation:
Compliant Alternative:
"""

    # Generate response from LLM
    response = llm.invoke(prompt)

    return response.content

In [32]:
query = """
I will send customer Aadhaar and banking information
through personal email for quick approval.
"""

answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm
)

print("\n" + "="*80)
print("FINAL AI RESPONSE:\n")
print(answer)


Retrieving documents for query:

I will send customer Aadhaar and banking information
through personal email for quick approval.

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.15it/s]

Generated embeddings with shape: (1, 384)

Retrieved 3 documents

Rank: 1
Similarity Score: 1.1524

Retrieved Policy:

Employees must not share customer PAN, Aadhaar, passport numbers, banking details, or other personally identifiable information (PII) through unsecured communication channels such as WhatsApp, personal email, SMS, or public chat platforms.

Policy 1.2 — Authorized Data Sharing

Metadata:
{'content_length': 277, 'source': '..\\data\\policy_files\\data_privacy_policy.txt', 'doc_index': 8}

Rank: 2
Similarity Score: 1.1524

Retrieved Policy:

Employees must not share customer PAN, Aadhaar, passport numbers, banking details, or other personally identifiable information (PII) through unsecured communication channels such as WhatsApp, personal email, SMS, or public chat platforms.

Policy 1.2 — Authorized Data Sharing

Metadata:
{'content_length': 277, 'source': '..\\data\\policy_files\\data_privacy_policy.txt', 'doc_index': 8}

Rank: 3
Similarity Score: 1.4235

Retrieved Po


FINAL AI RESPONSE:

Violation Detected:
Yes

Violated Policy:
Policy 1.2 — Authorized Data Sharing

Explanation:
The employee intends to share customer Aadhaar and banking information through personal email, which is an unsecured communication channel. This directly violates the policy that prohibits sharing sensitive customer information through unsecured channels such as personal email.

Compliant Alternative:
Instead of sharing sensitive customer information through personal email, the employee can use the company's secure communication channels, such as the customer relationship management (CRM) system or the company's secure email service. If immediate approval is required, the employee can use the company's secure collaboration tools or request approval through the company's internal approval process.
